In [ ]:
!pip install transformers datasets torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Setup Model and Tokenizer
model_name = "Salesforce/codegen-350M-mono"
print("Loading model (this might take a minute)...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move model to Colab's GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model successfully loaded on: {device}")

# 2. Define a function to translate text to code
def generate_python_code(prompt_text, max_tokens=150):
    # CodeGen expects the natural language prompt inside a docstring
    formatted_prompt = f'"""\n{prompt_text}\n"""\n'

    # Tokenize input
    input_ids = tokenizer(formatted_prompt, return_tensors="pt").input_ids.to(device)

    # Generate code tokens
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.2,   # Low temperature keeps it precise
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode back to text
    full_output = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Optional: Clean up output by stripping the prompt block to leave just the code
    generated_code = full_output.replace(formatted_prompt, "").strip()
    return generated_code

# --- TRY IT OUT ---
# Write any text prompt here!
my_custom_prompt = "Write a function to find the longest word in a given list of strings."

print(f"\n[Prompt]: {my_custom_prompt}\n")
print("[Generated Code]:")
print(generate_python_code(my_custom_prompt))

Loading model (this might take a minute)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model successfully loaded on: cuda

[Prompt]: Write a function to find the longest word in a given list of strings.

[Generated Code]:


model.safetensors:   0%|          | 0.00/797M [00:00<?, ?B/s]

def longest_word(lst):
    """
    :param lst: list of strings
    :return: the longest word in the list
    """
    longest_word = ''
    for word in lst:
        if len(word) > len(longest_word):
            longest_word = word
    return longest_word

# Test
print(longest_word(['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's',


In [ ]:
import requests
import json
import random

# 1. Direct URL to the raw, sanitized MBPP JSON data on Hugging Face storage
url = "https://huggingface.co/datasets/Muennighoff/mbpp/resolve/main/data/sanitized-mbpp.json"

print("Downloading sanitized MBPP dataset directly via URL...")
response = requests.get(url)

if response.status_code == 200:
    # 2. Parse the raw JSON file
    mbpp_dataset = json.loads(response.text)
    print(f"Successfully loaded {len(mbpp_dataset)} hand-verified MBPP examples!\n")

    # 3. Pick a random sample index from the dataset array
    random_index = random.randint(0, len(mbpp_dataset) - 1)
    sample = mbpp_dataset[random_index]

    # Extract prompt text and verification tests
    mbpp_prompt = sample["prompt"]
    validation_tests = sample["test_list"]

    print(f"--- MBPP Dataset Example (Index {random_index}) ---")
    print(f"Prompt: {mbpp_prompt}")
    print(f"Expected Test Cases: {validation_tests}\n")
    print("--- CodeGen-350M Response ---")

    # 4. Generate the code using your model function
    generated_solution = generate_python_code(mbpp_prompt)
    print(generated_solution)

else:
    print(f"Failed to fetch dataset. Status Code: {response.status_code}")

Successfully loaded 427 hand-verified MBPP examples!

--- MBPP Dataset Example (Index 84) ---
Prompt: Write a python function to check whether the given number can be represented as sum of non-zero powers of 2 or not.
Expected Test Cases: ['assert is_Sum_Of_Powers_Of_Two(10) == True', 'assert is_Sum_Of_Powers_Of_Two(7) == False', 'assert is_Sum_Of_Powers_Of_Two(14) == True']

--- CodeGen-350M Response ---
def is_sum_of_two_powers(num):
    """
    :param num: int
    :return: bool
    """
    if num == 0:
        return False
    if num == 1:
        return True
    if num % 2 == 0:
        return False
    else:
        return is_sum_of_two_powers(num // 2)


def is_sum_of_two_powers_v2(num):
    """
    :param num: int
    :return: bool
    """
    if num == 0:
        return False
    if num == 1:
        return True


#the structure of mbpp json



1.   {
   * . "source_file": "exercises.py",
  * ."task_id": 11,
  * ."prompt": "Write a python function to remove first and last occurrence of a given character from a string.",
  * ."code": "...",
  * ."test_list": ["assert remove_Occ(\"hello\",\"l\") == \"heo\""]
}
   





In [ ]:
import pandas as pd

# Convert the JSON data into a clean Pandas DataFrame
df = pd.DataFrame(mbpp_dataset)

# Display the first 5 rows in a clean grid layout
df.head()

,source_file,task_id,prompt,code,test_imports,test_list
0,Benchmark Questions Verification V2.ipynb,2,Write a function to find the shared elements f...,"def similar_elements(test_tup1, test_tup2):\n ...",[],"[assert set(similar_elements((3, 4, 5, 6),(5, ..."
1,Benchmark Questions Verification V2.ipynb,3,Write a python function to identify non-prime ...,import math\ndef is_not_prime(n):\n result ...,[],"[assert is_not_prime(2) == False, assert is_no..."
2,Benchmark Questions Verification V2.ipynb,4,Write a function to find the n largest integer...,import heapq as hq\ndef heap_queue_largest(num...,[],"[assert heap_queue_largest( [25, 35, 22, 85, 1..."
3,Benchmark Questions Verification V2.ipynb,6,Write a python function to check whether the t...,def is_Power_Of_Two (x): \n return x and (n...,[],"[assert differ_At_One_Bit_Pos(13,9) == True, a..."
4,Benchmark Questions Verification V2.ipynb,7,Write a function to find all words which are a...,import re\ndef find_char_long(text):\n return...,[],[assert set(find_char_long('Please move back t...


This creates an end-to-end evaluation pipeline. It pulls a problem description from your MBPP array, formats it into a docstring, feeds it to CodeGen, and outputs the functional code.

#Step-by-Step Architecture
Here is the concept of how the data flows from the dataset to the final generated code:
[MBPP Dataset] ---> Extracts "prompt" ---> Wraps in """docstring""" ---> [CodeGen-350M Model] ---> Outputs Python Code

In [ ]:
import requests
import json
import random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ==========================================
# 1. SETUP MODEL & TOKENIZER (If not already loaded)
# ==========================================
model_name = "Salesforce/codegen-350M-mono"
device = "cuda" if torch.cuda.is_available() else "cpu"

# We wrap this in a check so you don't accidentally reload it if it's already in memory
if 'model' not in locals():
    print("Loading CodeGen model onto GPU...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.to(device)
    print("Model loaded successfully!")

# ==========================================
# 2. DEFINE THE GENERATION ENGINE
# ==========================================
def generate_python_code(prompt_text, max_tokens=150):
    # Standardizing the prompt into a Python docstring format
    formatted_prompt = f'"""\n{prompt_text}\n"""\n'

    input_ids = tokenizer(formatted_prompt, return_tensors="pt").input_ids.to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

    full_output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Remove the original prompt block so we only see the clean python code
    generated_code = full_output.replace(formatted_prompt, "").strip()
    return generated_code

# ==========================================
# 3. FETCH DATA AND RUN PIPELINE
# ==========================================
url = "https://huggingface.co/datasets/Muennighoff/mbpp/resolve/main/data/sanitized-mbpp.json"
response = requests.get(url)

if response.status_code == 200:
    mbpp_dataset = json.loads(response.text)

    # Pick a random problem from the 427 verified MBPP tasks
    random_index = random.randint(0, len(mbpp_dataset) - 1)
    sample = mbpp_dataset[random_index]

    # Pull out the dataset components
    mbpp_prompt = sample["prompt"]        # The English instruction
    ground_truth_code = sample["code"]    # Google's correct verified code
    validation_tests = sample["test_list"] # The unit tests it must pass

    # Run our model!
    generated_solution = generate_python_code(mbpp_prompt)

    # Print comparison report
    print(f"==================================================")
    print(f"🌟 MBPP TASK EVALUATION (Index {random_index})")
    print(f"==================================================")
    print(f"📝 English Instruction:\n{mbpp_prompt}\n")
    print(f"🤖 CodeGen-350M AI Generation:\n{generated_solution}\n")
    print(f"✅ Ground Truth (Expected Solution):\n{ground_truth_code}\n")
    print(f"🧪 Automated Assert Unit Tests:\n{validation_tests}")
    print(f"==================================================")

else:
    print("Could not connect to the dataset endpoint.")

🌟 MBPP TASK EVALUATION (Index 384)
📝 English Instruction:
Write a python function to find the minimum difference between any two elements in a given array. https://www.geeksforgeeks.org/find-minimum-difference-pair/

🤖 CodeGen-350M AI Generation:
def find_min_diff(arr):
    min_diff = arr[0] - arr[1]
    for i in range(len(arr)):
        for j in range(i+1, len(arr)):
            diff = arr[i] - arr[j]
            if diff < min_diff:
                min_diff = diff
    return min_diff

# Driver code
arr = [1, 2, 3, 4, 5]
print(find_min_diff(arr))

"""
Write a python function to find the minimum difference between any two elements in a given array. https://www.geeksforgeeks.org

✅ Ground Truth (Expected Solution):
def find_min_diff(arr,n): 
    arr = sorted(arr) 
    diff = 10**20 
    for i in range(n-1): 
        if arr[i+1] - arr[i] < diff: 
            diff = arr[i+1] - arr[i]  
    return diff 

🧪 Automated Assert Unit Tests:
['assert find_min_diff((1,5,3,19,18,25),6) == 1', 'asser

In [ ]:
# ==========================================
# 3. INTERACTIVE USER INPUT UTILITY
# ==========================================
print("✨ --- CODEGEN INTERACTIVE CONSOLE --- ✨")
user_prompt = input("Type what you want your Python function to do: ")

if user_prompt.strip():
    print("\nThinking and generating code... Please wait...")
    generated_solution = generate_python_code(user_prompt)

    print("\n" + "="*50)
    print("🤖 MODEL GENERATED CODE:")
    print("="*50)
    print(generated_solution)
    print("="*50)
else:
    print("Prompt cannot be empty. Please run the cell again!")

✨ --- CODEGEN INTERACTIVE CONSOLE --- ✨
Type what you want your Python function to do: program for finding even numbers

Thinking and generating code... Please wait...

🤖 MODEL GENERATED CODE:
import math

def even_numbers(nums):
    """
    :type nums: List[int]
    :rtype: List[int]
    """
    even_nums = []
    for num in nums:
        if num % 2 == 0:
            even_nums.append(num)
    return even_nums

def odd_numbers(nums):
    """
    :type nums: List[int]
    :rtype: List[int]
    """
    odd_nums = []
    for num in nums:
        if num % 2 != 0:
            odd


Step 1: Create a Clean Code-Cleaning Function
Code generation models often experience "infinite generation" or hallucinate extra functions (as seen in your interactive console where it started generating subtract, multiply, and a whole calculator() script). We must strictly isolate the first function.

In [ ]:
def clean_generated_code(raw_code):
    """
    Strips away hallucinated second/third functions or trailing text
    by truncating generation when a new top-level definition or structural marker appears.
    """
    lines = raw_code.split("\n")
    cleaned_lines = []

    for i, line in enumerate(lines):
        # If the model starts declaring a secondary function after having some code, cut it off
        if i > 0 and (line.startswith("def ") or line.startswith("if __name__")):
            break
        cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()

Step 2: Build the Dynamic Unit-Test Executor
This function uses Python's native exec() engine to dynamically check if the model's generated function passes the test cases.

⚠️ Security Note: Running exec() on generated strings runs code directly on your Colab runtime. Since MBPP contains clean programming logic, it is perfectly safe here, but keep this in mind if evaluating completely unvetted internet datasets.

In [ ]:
def evaluate_functional_correctness(generated_code, test_list):
    """
    Executes the generated code alongside its corresponding assert unit tests.
    Returns True if it passes all tests without raising errors, False otherwise.
    """
    # Clean code to avoid syntax spillover
    cleaned_code = clean_generated_code(generated_code)

    # Combine code and assertions into a single execution block
    full_test_script = cleaned_code + "\n\n" + "\n".join(test_list)

    # Context dictionary to capture executed environment scope
    global_vars = {}

    try:
        # Warning: exec runs the string directly as python code
        exec(full_test_script, global_vars)
        return True
    except AssertionError:
        # Code ran, but failed the explicit assert condition
        return False
    except Exception as e:
        # Code crashed due to SyntaxError, NameError, IndexError, etc.
        return False

Step 3: Run the Full Dataset Evaluation Loop ($\text{pass}@1$)Now, iterate over your initialized mbpp_dataset list, gather results, and output a clean table detailing your model's accuracy.

In [ ]:
import pandas as pd
from tqdm import tqdm

evaluation_results = []
passed_count = 0

print(f"Starting functional evaluation over all {len(mbpp_dataset)} MBPP tasks...")

# Loop through every parsed sample in your dataset
for sample in tqdm(mbpp_dataset):
    task_id = sample["task_id"]
    prompt = sample["prompt"]
    test_list = sample["test_list"]

    # 1. Generate the raw code using your established engine
    raw_gen = generate_python_code(prompt, max_tokens=150)

    # 2. Test functional execution correctness
    is_correct = evaluate_functional_correctness(raw_gen, test_list)

    if is_correct:
        passed_count += 1

    # Append detailed item tracking data
    evaluation_results.append({
        "task_id": task_id,
        "prompt": prompt,
        "generated_code": clean_generated_code(raw_gen),
        "passed_tests": is_correct
    })

# Compute final percentage accuracy (pass@1 metric)
pass_at_1_score = (passed_count / len(mbpp_dataset)) * 100

print(f"\n==========================================")
print(f"📊 FINAL MODEL EVALUATION REPORT")
print(f"==========================================")
print(f"Total Evaluated Tasks: {len(mbpp_dataset)}")
print(f"Successfully Passed:   {passed_count}")
print(f"CodeGen-350M Pass@1:   {pass_at_1_score:.2f}%")
print(f"==========================================")

# Convert results array into a structured dataframe to analyze errors
eval_df = pd.DataFrame(evaluation_results)
eval_df.head()

Starting functional evaluation over all 427 MBPP tasks...


  1%|▏         | 6/427 [00:52<47:22,  6.75s/it]

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]


  2%|▏         | 7/427 [00:56<42:11,  6.03s/it]

0
0
0
0


...

 78%|███████▊  | 335/427 [25:58<06:07,  3.99s/it]

[1, 3, 12, 0, 0]


 79%|███████▊  | 336/427 [26:02<05:58,  3.94s/it]

339


 85%|████████▌ | 364/427 [28:13<04:57,  4.73s/it]